# NOTEARS noise_ratio benchmark

? ER DAG ????? `noise_ratio` ? NOTEARS ????

???? trial ???? `(noise_ratio, d, n_samples)` ??? summary?????? `experiments/results/`?

In [9]:
import logging
import os
import sys
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "synthetic_dataset.py").exists() and (path / "MEC.py").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from MEC import find_v_structures, get_skeleton, is_in_markov_equiv_class
from synthetic_dataset import SyntheticDataset

_previous_disable_level = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    from castle.algorithms import Notears
finally:
    logging.disable(_previous_disable_level)

try:
    TOOLBOX_ROOT = REPO_ROOT / "toolbox"
    if str(TOOLBOX_ROOT) not in sys.path:
        sys.path.append(str(TOOLBOX_ROOT))
    from cdt.metrics import SHD_CPDAG as cdt_shd_cpdag
except Exception:
    cdt_shd_cpdag = None

print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = C:\Users\super\DAG


In [ ]:
CFG = {
    "trials": 1,
    "seed": 42,
    "d_list": [50],
    "n_list": [20000],
    "degree": 1.0,
    "noise_type": "gaussian_nv",
    "noise_ratios": [16],
    "B_scale": 1.0,
    "notears_lambda1": 0.1,
    "notears_loss_type": "l2",
    "notears_threshold": 0.3,
    "verbose_notears": False,
    "outdir": REPO_ROOT / "experiments" / "results",
    "tag": "notears_noise_ratio_benchmark",
}

CFG

{'trials': 1,
 'seed': 42,
 'd_list': [50],
 'n_list': [20000],
 'degree': 1.0,
 'noise_type': 'gaussian_nv',
 'noise_ratios': [16],
 'B_scale': 2.0,
 'notears_lambda1': 0.1,
 'notears_loss_type': 'l2',
 'notears_threshold': 0.3,
 'verbose_notears': False,
 'outdir': WindowsPath('C:/Users/super/DAG/experiments/results'),
 'tag': 'notears_noise_ratio_benchmark'}

In [11]:
@contextmanager
def maybe_suppress_notears_logs(verbose: bool):
    if verbose:
        yield
        return

    previous_disable_level = logging.root.manager.disable
    logging.disable(logging.INFO)
    try:
        yield
    finally:
        logging.disable(previous_disable_level)


@dataclass
class TrialRow:
    noise_ratio: float
    d: int
    n_samples: int
    trial_id: int
    seed: int
    algorithm: str
    status: str
    mec_match: int
    shd: float
    cpdag_shd: float
    precision: float
    recall: float
    f1: float
    n_edges_true: int
    n_edges_est: int
    runtime_sec: float
    message: str


def weight_to_binary_adj(W: np.ndarray, threshold: float = 0.0) -> np.ndarray:
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    """Pairwise SHD: reversing one edge counts as one error."""
    G_true = np.asarray(G_true, dtype=int)
    G_est = np.asarray(G_est, dtype=int)
    d = G_true.shape[0]
    dist = 0
    for i in range(d):
        for j in range(i + 1, d):
            if G_true[i, j] != G_est[i, j] or G_true[j, i] != G_est[j, i]:
                dist += 1
    return float(dist)


def precision_recall_f1(G_true: np.ndarray, G_est: np.ndarray) -> Tuple[float, float, float]:
    true_edge = G_true == 1
    pred_edge = G_est == 1

    tp = int(np.sum(true_edge & pred_edge))
    fp = int(np.sum((~true_edge) & pred_edge))
    fn = int(np.sum(true_edge & (~pred_edge)))

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return float(precision), float(recall), float(f1)


def cpdag_shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    if cdt_shd_cpdag is not None:
        try:
            return float(cdt_shd_cpdag(G_true.astype(int), G_est.astype(int)))
        except Exception:
            pass

    skel_true = get_skeleton(G_true)
    skel_est = get_skeleton(G_est)
    skel_diff = int(np.sum(np.abs(skel_true - skel_est)) // 2)
    v_diff = len(find_v_structures(G_true).symmetric_difference(find_v_structures(G_est)))
    return float(skel_diff + v_diff)


def make_failure_row(noise_ratio, d, n_samples, trial_id, seed, message) -> TrialRow:
    return TrialRow(
        noise_ratio=noise_ratio,
        d=d,
        n_samples=n_samples,
        trial_id=trial_id,
        seed=seed,
        algorithm="notears",
        status="failed",
        mec_match=0,
        shd=np.nan,
        cpdag_shd=np.nan,
        precision=np.nan,
        recall=np.nan,
        f1=np.nan,
        n_edges_true=0,
        n_edges_est=0,
        runtime_sec=np.nan,
        message=message,
    )

In [12]:
def run_trial(noise_ratio: float, d: int, n_samples: int, trial_id: int, seed: int, cfg: dict) -> TrialRow:
    dataset = SyntheticDataset(
        n=n_samples,
        d=d,
        graph_type="ER",
        degree=cfg["degree"],
        noise_type=cfg["noise_type"],
        noise_ratio=noise_ratio,
        B_scale=cfg["B_scale"],
        seed=seed,
    )
    X = dataset.X
    G_true = weight_to_binary_adj(dataset.B, threshold=0.0)

    model = Notears(
        lambda1=cfg["notears_lambda1"],
        loss_type=cfg["notears_loss_type"],
        w_threshold=cfg["notears_threshold"],
    )

    t0 = time.perf_counter()
    with maybe_suppress_notears_logs(cfg.get("verbose_notears", False)):
        model.learn(X)
    runtime_sec = time.perf_counter() - t0

    G_est = np.asarray(model.causal_matrix, dtype=int)
    np.fill_diagonal(G_est, 0)

    precision, recall, f1 = precision_recall_f1(G_true, G_est)
    return TrialRow(
        noise_ratio=noise_ratio,
        d=d,
        n_samples=n_samples,
        trial_id=trial_id,
        seed=seed,
        algorithm="notears",
        status="ok",
        mec_match=int(is_in_markov_equiv_class(G_true, G_est)),
        shd=shd_score(G_true, G_est),
        cpdag_shd=cpdag_shd_score(G_true, G_est),
        precision=precision,
        recall=recall,
        f1=f1,
        n_edges_true=int(G_true.sum()),
        n_edges_est=int(G_est.sum()),
        runtime_sec=float(runtime_sec),
        message="",
    )


def summarize(df_trials: pd.DataFrame) -> pd.DataFrame:
    ok = df_trials[df_trials["status"] == "ok"].copy()
    if ok.empty:
        return pd.DataFrame()

    summary = (
        ok.groupby(["noise_ratio", "d", "n_samples", "algorithm"], as_index=False)
        .agg(
            trials=("trial_id", "count"),
            mec_match_mean=("mec_match", "mean"),
            shd_mean=("shd", "mean"),
            shd_std=("shd", "std"),
            cpdag_shd_mean=("cpdag_shd", "mean"),
            cpdag_shd_std=("cpdag_shd", "std"),
            precision_mean=("precision", "mean"),
            recall_mean=("recall", "mean"),
            f1_mean=("f1", "mean"),
            n_edges_true_mean=("n_edges_true", "mean"),
            n_edges_est_mean=("n_edges_est", "mean"),
            runtime_sec_mean=("runtime_sec", "mean"),
            runtime_sec_std=("runtime_sec", "std"),
        )
        .sort_values(["d", "n_samples", "noise_ratio"])
        .reset_index(drop=True)
    )

    failures = (
        df_trials[df_trials["status"] != "ok"]
        .groupby(["noise_ratio", "d", "n_samples", "algorithm"], as_index=False)
        .size()
        .rename(columns={"size": "failures"})
    )
    summary = summary.merge(failures, on=["noise_ratio", "d", "n_samples", "algorithm"], how="left")
    summary["failures"] = summary["failures"].fillna(0).astype(int)
    return summary

In [13]:
def run_benchmark(cfg: dict):
    outdir = Path(cfg["outdir"])
    outdir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(cfg["seed"])
    rows = []

    print("NOTEARS noise_ratio benchmark")
    print(f"  d_list       : {cfg['d_list']}")
    print(f"  n_list       : {cfg['n_list']}")
    print(f"  trials       : {cfg['trials']}")
    print(f"  degree       : {cfg['degree']}")
    print(f"  noise_type   : {cfg['noise_type']}")
    print(f"  noise_ratios : {cfg['noise_ratios']}")
    print(f"  lambda1      : {cfg['notears_lambda1']}")
    print(f"  threshold    : {cfg['notears_threshold']}")

    for d in cfg["d_list"]:
        for n_samples in cfg["n_list"]:
            seeds = rng.integers(0, 10**9, size=cfg["trials"])
            for trial_idx, seed_raw in enumerate(seeds, start=1):
                seed = int(seed_raw)
                for noise_ratio in cfg["noise_ratios"]:
                    try:
                        row = run_trial(noise_ratio, d, n_samples, trial_idx, seed, cfg)
                        print(
                            f"[notears] ratio={noise_ratio:g} d={d} n={n_samples} "
                            f"trial={trial_idx} mec={row.mec_match} "
                            f"shd={row.shd:.0f} cpdag_shd={row.cpdag_shd:.0f} "
                            f"rt={row.runtime_sec:.3f}s"
                        )
                    except Exception as exc:
                        row = make_failure_row(noise_ratio, d, n_samples, trial_idx, seed, str(exc))
                        print(f"[SKIP] ratio={noise_ratio:g} d={d} n={n_samples} trial={trial_idx}: {exc}")
                    rows.append(row)

    df_trials = pd.DataFrame([asdict(r) for r in rows])
    df_summary = summarize(df_trials)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    trials_path = outdir / f"{cfg['tag']}_trials_{ts}.csv"
    summary_path = outdir / f"{cfg['tag']}_summary_{ts}.csv"
    latest_trials_path = outdir / f"{cfg['tag']}_trials.csv"
    latest_summary_path = outdir / f"{cfg['tag']}_summary.csv"

    df_trials.to_csv(trials_path, index=False)
    df_summary.to_csv(summary_path, index=False)
    df_trials.to_csv(latest_trials_path, index=False)
    df_summary.to_csv(latest_summary_path, index=False)

    print("\nSaved:")
    for path in [trials_path, summary_path, latest_trials_path, latest_summary_path]:
        print(f"  - {path}")

    return df_trials, df_summary, {
        "trials": trials_path,
        "summary": summary_path,
        "latest_trials": latest_trials_path,
        "latest_summary": latest_summary_path,
    }


df_trials, df_summary, saved_paths = run_benchmark(CFG)

NOTEARS noise_ratio benchmark
  d_list       : [50]
  n_list       : [20000]
  trials       : 1
  degree       : 1.0
  noise_type   : gaussian_nv
  noise_ratios : [16]
  lambda1      : 0.1
  threshold    : 0.3
[notears] ratio=16 d=50 n=20000 trial=1 mec=1 shd=0 cpdag_shd=0 rt=728.393s

Saved:
  - C:\Users\super\DAG\experiments\results\notears_noise_ratio_benchmark_trials_20260427_012104.csv
  - C:\Users\super\DAG\experiments\results\notears_noise_ratio_benchmark_summary_20260427_012104.csv
  - C:\Users\super\DAG\experiments\results\notears_noise_ratio_benchmark_trials.csv
  - C:\Users\super\DAG\experiments\results\notears_noise_ratio_benchmark_summary.csv


In [ ]:
display(df_summary)

failed = df_trials[df_trials["status"] != "ok"]
if not failed.empty:
    display(failed[["noise_ratio", "d", "n_samples", "trial_id", "seed", "message"]])

In [ ]:
plot_df = df_summary.copy()
if not plot_df.empty:
    metrics = ["mec_match_mean", "shd_mean", "cpdag_shd_mean", "precision_mean", "recall_mean", "f1_mean", "runtime_sec_mean"]
    ncols = 2
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows), squeeze=False)

    for ax, metric in zip(axes.ravel(), metrics):
        for (d, n_samples), grp in plot_df.groupby(["d", "n_samples"]):
            grp = grp.sort_values("noise_ratio")
            ax.plot(grp["noise_ratio"], grp[metric], marker="o", label=f"d={d}, n={n_samples}")
        ax.set_xlabel("noise_ratio")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for ax in axes.ravel()[len(metrics):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()